# 🧠 Semantic Search Over Scouting Reports with Cohere + Databricks Vector Search

This notebook demonstrates how to run **semantic similarity queries** over scouting reports using **Cohere embeddings** and **Databricks Vector Search**.

We explore two core use cases:

### 🔍 1. Natural Language Search
Use a descriptive phrase (e.g. _"elite bat speed but struggles with offspeed pitches"_) to find relevant players based on the content of their reports.

### 🧠 2. Report-to-Report Similarity
Embed a full scouting report for a given player and return the most semantically similar players in the index — useful for comp search and player discovery.

This notebook uses:
- A Delta Sync–powered vector index of chunked scouting reports
- External model serving with **Cohere’s `embed-english-v3`**
- Native search via `similarity_search()` from the Python SDK
- Joins back to the full report for readable output


In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.3
    Not uninstalling protobuf at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-2d4d780a-cceb-4de4-9271-ccbd7085816b
    Can't uninstall 'protobuf'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.69.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.7 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType
from pyspark.sql import Row
import pyspark.sql.functions as F

# Initialize the Vector Search client
vsc = VectorSearchClient(disable_notice=True)

In [0]:
# Environment Variables
CATALOG = "alexander_booth"
SCHEMA = "cohere_demo"
TABLE = "fangraphs_mlb_scouting_reports"
TABLE_CHUNKS = "fangraphs_mlb_scouting_reports_chunked"

# Vector Store Environment Variables
INDEX_NAME = "fangraphs_mlb_scouting_reports_index"
EMBEDDING_MODEL = "cohere_embed_english_v3"  # your model serving endpoint
VECTOR_SEARCH_ENDPOINT = "vs_scouting_reports_demo"
PRIMARY_KEY = "primary_key"
SOURCE_COLUMN = "content_chunk"

# 🔍 Semantic Search from Natural Language Query

In this section, we use a **natural language phrase** to find players with similar scouting reports. The phrase is embedded using Cohere’s `search_query` mode, and then searched against our vector index of report chunks.

This enables use cases like:
- "Show me players with elite bat speed but struggle with offspeed pitches"
- "Find hitters with power but poor plate discipline"
- "Find pitchers with a good fastball but poor command"


In [0]:
def search_similar_players_by_text(index, query_text: str, top_k: int = 5):
    """
    Run a semantic search on a vector index using a query string, and return the top similar players.

    Parameters:
    - index: VectorSearchIndex object
    - query_text: Natural language query string
    - top_k: Number of top players to return (default 5)

    Returns:
    - final_df: Spark DataFrame with top_k similar players and their scouting reports
    """
    
    # Step 1: Run semantic search
    results = index.similarity_search(
        query_text=query_text,
        columns=["primary_key", "playerName", "ID", "Season", "chunk_index", "content_chunk"],
        num_results=100
    )

    # Step 2: Parse results
    rows = results["result"]["data_array"]
    parsed_rows = [
        Row(
            primary_key=row[0],
            playerName=row[1],
            ID=int(row[2]),
            Season=int(row[3]),
            chunk_index=int(row[4]),
            content_chunk=row[5],
            score=float(row[6])
        )
        for row in rows
    ]

    schema = StructType([
        StructField("primary_key", StringType()),
        StructField("playerName", StringType()),
        StructField("ID", IntegerType()),
        StructField("Season", IntegerType()),
        StructField("chunk_index", IntegerType()),
        StructField("content_chunk", StringType()),
        StructField("score", FloatType())
    ])

    # Step 3: Create DataFrame and temp view
    results_df = spark.createDataFrame(parsed_rows, schema=schema)
    results_df.createOrReplaceTempView("semantic_chunks")

    # Step 4: Aggregate top players
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW top_players AS
        SELECT
            ID,
            playerName,
            Season,
            MAX(score) AS top_chunk_score
        FROM semantic_chunks
        GROUP BY ID, playerName, Season
        ORDER BY top_chunk_score DESC
        LIMIT {top_k}
    """)

    # Step 5: Join back to full report
    final_df = spark.sql(f"""
        SELECT
            orig.ID,
            orig.playerName,
            orig.Season,
            orig.Combined_Scouting_Report,
            top.top_chunk_score
        FROM top_players AS top
        JOIN {CATALOG}.{SCHEMA}.{TABLE} AS orig
          ON top.ID = orig.ID AND top.Season = orig.Season
        ORDER BY top.top_chunk_score DESC
    """)

    return final_df


In [0]:
# Build the full index name
FULL_INDEX_NAME = f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"

# Retrieve the index object
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=FULL_INDEX_NAME
)

QUERY = "Elite bat speed but has trouble with offspeed pitches"

final_df = search_similar_players_by_text(
    index=index,
    query_text=QUERY
)

display(final_df)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


ID,playerName,Season,Combined_Scouting_Report,top_chunk_score
2424820,Jac Caglianone,2025,"TLDR: Caglianone was a two-way college player with elite power and arm strength. If his plate discipline and feel for contact improve as he focuses on hitting, he's going to explode. Full Report: The freakishly talented Caglianone presented a serious conundrum to teams picking atop the 2024 draft. He was perhaps as talented and toolsy a college player as there has been in the last decade or so, wielding elite raw power and, for parts of his career, elite fastball velocity, but Caglianone's plate discipline and command were both so lacking that they threatened his entire profile on both sides of the ball. He slugged .738 and hit 33 homers as a sophomore, then slashed .419/.544/.875 with 35 bombs in his draft year, but long levers and poor ball/strike recognition (his chase rates approached 40% in college) caused Cags to whiff quite a bit more than his 8.2% junior year K% might otherwise indicate. His swing, which features a very aggressive hand load way up and away from his body, takes advantage of Caglianone's prodigious athleticism to generate nutty power, but it might benefit from being toned down, not only to help stop Cags from striking out too much, but so that he doesn't drive the ball into the ground so often, which was an issue after the draft. Caglianone looks as impressive in his uniform as anyone in pro baseball (his thighs are practically bursting out of his pants), and he's strong enough to do damage with a simpler operation. These blemishes create risk, but the notion that Caglianone will polish his hit tool and feel for the strike zone once he's been allowed to focus on hitting in pro ball makes him a very exciting prospect. There's also an exciting fallback option on the mound if strikeouts turn out to be a problem, though that comes with its own risk. Caglianone has an ideal pitcher's frame at a hulking 6-foot-5. He's limber and loose and powers down the mound with ease. His fastball sat 97-100 in 2023, and he has a great changeup. But Caglianone's velocity plummeted in 2024, and he only averaged 92-93 mph from May until the end of the season. His two breaking balls (an upper-80s slider/cutter and a low-80s slurve) are also wildly inconsistent, as is his command and overall feel for pitching. A return to the mound should probably only be explored if Caglianone underwhelms as a hitter for the next couple of seasons. His ceiling as an everyday hitter is big enough that it should be his/Kansas City's focus until it becomes clear that it isn't going to work. This guy has 35 homer upside.",0.5171014
2425549,Dylan Crews,2025,"TLDR: Crews has big power and speed, and altered his position in the batter's box last year to better deal with fastballs around his hands. Full Report: Crews went wire-to-wire as one of the best 2023 draft prospects, if not the best. He was the top unsigned high schooler from the 2020 class, a toolshed who swung and missed on the summer showcase circuit more than teams felt comfortable with. He ended up at LSU rather than in pro ball, and from day one was among the toolsiest and most productive college hitters for three straight years. He slashed .380/.498/.689 and had more walks than strikeouts during his tenure at LSU. Crews' first full pro season also went well. He slashed .270/.342/.451 split between Double-A Harrisburg and Triple-A Rochester, then was called up at the end of August to get his feet wet without exhausting rookie eligibility. His underlying data was much better than his surface-level stats during that stretch. Crews can punish you to all fields. He'll get extended on fastballs away from him and crush them the opposite way, and he can also turn on slower pitches on the middle two-thirds of the plate and hit some titanic blasts to left. His best swings feature unbelievable verve and explosion, with Crews' lower body usage evoking Mike Trout. At times, the depth of Crews' load will leave 

# 🧠 Semantic Search from a Scouting Report

Here, we select a real player's full scouting report, embed it using Cohere, and then search for **semantically similar player profiles**.

This enables:
- Player comparison / comp search
- Scouting assistant workflows ("who does this guy remind you of?")
- Building similarity-driven dashboards and ranking tools


In [0]:
def find_similar_to_player_report(index, player_name: str, season: int = None, top_k: int = 5):

    # Step 1: Lookup and truncate the report
    scouting_df = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}")
    filter_df = scouting_df.filter(F.col("playerName") == player_name)
    if season:
        filter_df = filter_df.filter(F.col("Season") == season)

    result = filter_df.select("Combined_Scouting_Report").limit(1).collect()
    if not result:
        raise ValueError(f"No scouting report found for {player_name} (season={season})")

    report_text = result[0]["Combined_Scouting_Report"][:2048]  # truncate

    # Step 2: Run semantic similarity search directly using the report text
    results = index.similarity_search(
        query_text=report_text,
        columns=["primary_key", "playerName", "ID", "Season", "chunk_index", "content_chunk"],
        num_results=100
    )

    # Step 3: Parse response into Spark DataFrame
    rows = results["result"]["data_array"]
    parsed_rows = [
        Row(
            primary_key=row[0],
            playerName=row[1],
            ID=int(row[2]),
            Season=int(row[3]),
            chunk_index=int(row[4]),
            content_chunk=row[5],
            score=float(row[6])
        )
        for row in rows
    ]

    schema = StructType([
        StructField("primary_key", StringType()),
        StructField("playerName", StringType()),
        StructField("ID", IntegerType()),
        StructField("Season", IntegerType()),
        StructField("chunk_index", IntegerType()),
        StructField("content_chunk", StringType()),
        StructField("score", FloatType())
    ])

    semantic_df = spark.createDataFrame(parsed_rows, schema=schema)
    semantic_df.createOrReplaceTempView("semantic_chunks")

    # Step 4: Aggregate scores to player level
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW top_players AS
        SELECT
            ID,
            playerName,
            Season,
            MAX(score) AS similarity
        FROM semantic_chunks
        GROUP BY ID, playerName, Season
        ORDER BY similarity DESC
        LIMIT {top_k}
    """)

    # Step 5: Join with full reports
    final_df = spark.sql(f"""
        SELECT
            p.playerName,
            p.Season,
            r.Combined_Scouting_Report,
            p.similarity
        FROM top_players p
        JOIN {CATALOG}.{SCHEMA}.{TABLE} r
          ON p.ID = r.ID AND p.Season = r.Season
        ORDER BY p.similarity DESC
    """)

    return final_df

In [0]:
final_df = find_similar_to_player_report(
    index=index,
    player_name='Sebastian Walcott',
    season=2025
)

display(final_df)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


playerName Season Combined_Scouting_Report similarity Sebastian Walcott 2025 TLDR:
If you're looking for the next Fernando Tatis Jr., this is your most likely candidate.

Full Report:
Walcott was seen some in the U.S. during the fall of 2021, and he leapt off the field because of his physicality and power. The Rangers international scouting group was originally connected to a player named Camilo Diaz for a rumored bonus of $3 million, but seemingly diverted to Walcott around this time (Diaz later signed with Houston). Walcott signed in January of 2023 and immediately stood apart from his peers on the backfields because of his size and bat speed. He went to the D.R. for the start of the 2023 DSL season, but was quickly promoted to Arizona and lit up the ACL up for a month (he was hondo'd at this time) before opponents realized he couldn't recognize a breaking ball. He ended up having a .273/.325/.524 line in Arizona with terrifying peripherals, including a 6.4% walk rate and 32.5% strikeout rate. In 2024, the Rangers sent Walcott, who turned 18 during spring training, straight to High-A Hickory. Amazingly, he held his own there, slashing .261/.342/.443 with 10.6% walks (sigh of relief) and 25.5% strikeouts before earning a promotion to Double-A Frisco for the final week of the season. 
As you are reading this Top 100 blurb, Walcott is still just 18 years old. He has immense physical ability, a cathedral ceiling, and the long-term athletic projection to be very good for a very long time. He looks like a future NFL wide receiver in his uniform and has elite hand speed in the batter's box, which he generates with shockingly little effort for a hitter his age. There are some mechanical flaws and nits to pick with Walcott's swing, but when you're this talented, these tend not to matter. Walcott's front side often opens up toward third base when he swings, leaving him very vulnerable to sliders away from him. His tendency to chase and miss against secondary stuff is still kind of scary, but his fastball swing decisions are very good, and overall, his baseline chase and two-strike chase were about the big league average in 2024. His bat path slopes down and tends to swat the baseball into the ground, unless Walcott is making contact way out in front of the plate, at which point those of you in the parking lot need to keep your head on a swivel. Despite his issues, Walcott is generating plus big league peak exit velos as a freaking 18-year-old, and just had a 125 wRC+ season in the mid-minors even though the paint on many aspects of his game is still wet. His stride direction might require eventual adjustment, but rather than intervene right now, I think it's fine to see if Walcott's timing and feel to hit come naturally once he's challenged. 
By the end of the season, Walcott had also leveled up on defense. He lacks the footspeed and twitch of most big league shortstops, but his ability to bend at his size is amazing, and Walcott's short arm stroke (reminiscent of Manny Machado's) allows him to effortlessly send the baseball wherever it needs to go. His actions and exchange on slow rollers have also improved. By the end of the season, Walcott (who still has some accuracy hiccups) did not seem sped up or flustered by the speed of the game on defense like he had early in the season. Walcott's ceiling as a power hitter is as big as any prospect in the minors, and of the players close to him, he pretty comfortably has the best chance at remaining at what is arguably the most valuable position in baseball. This guy is easily the odds on favorite to be the top prospect in baseball a year from now, and he's a grade of breaking ball recognition away from being a 70 FV talent. 0.7366332 Hurston Waldrep 2025 TLDR:
Waldrep's high-octane delivery and "round down" fastball might force him into the bullpen, but his splitter should help make him an impact late-inning arm even if that's the case.

Full Report:
Waldrep barely pitched as a freshman at 

---

✅ That wraps up this technical walkthrough on powering semantic search with **Cohere embeddings** and **Databricks Vector Search**.

We covered how to:
- Load and chunk real scouting reports
- Embed them using a Cohere model served through AWS Bedrock
- Create a live, queryable vector index with Delta Sync
- Perform similarity search using both natural language and existing reports

---


In a follow-up demo, we’ll show how to combine this vector store with a large language model in a **Retrieval-Augmented Generation (RAG)** setup.

Stay tuned — semantic search is just the first step.
